# Shared-coverage HDF5 walkthrough

This standalone notebook explores one `*.shared_coverage.h5` file. It does not import any RiboFlow Paper modules and does not need the BAM files that generated the HDF5. It demonstrates the file hierarchy, provenance, transcript lookup by **gene ID**, the shared transcript coordinate, region and exon annotations, and genome-versus-transcriptome coverage.

Requirements: `h5py`, `numpy`, `pandas`, and `matplotlib`. Set the `RIBOFLOW_COVERAGE_H5` environment variable or edit `COVERAGE_H5` below.

In [ ]:
from pathlib import Path
import json
import os

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def default_coverage_path():
    configured = os.environ.get("RIBOFLOW_COVERAGE_H5")
    if configured:
        return Path(configured).expanduser().resolve()
    candidates = [
        Path.cwd() / "results/coverage/HeLa.shared_coverage.h5",
        Path.cwd().parent / "results/coverage/HeLa.shared_coverage.h5",
    ]
    return next((path.resolve() for path in candidates if path.exists()), candidates[0].resolve())

COVERAGE_H5 = default_coverage_path()
if not COVERAGE_H5.exists():
    raise FileNotFoundError(
        f"Coverage file not found: {COVERAGE_H5}\n"
        "Edit COVERAGE_H5 or set RIBOFLOW_COVERAGE_H5 to a *.shared_coverage.h5 file."
    )
COVERAGE_H5

## 1. Open the file and inspect its hierarchy

An HDF5 file resembles a small filesystem: groups behave like directories, datasets behave like typed arrays, and attributes hold metadata.

In [ ]:
h5 = h5py.File(COVERAGE_H5, "r")

def print_tree(handle):
    def visit(name, obj):
        depth = name.count("/")
        prefix = "    " * depth
        if isinstance(obj, h5py.Group):
            print(f"{prefix}{name}/")
        else:
            print(f"{prefix}{name}: shape={obj.shape}, dtype={obj.dtype}")
    handle.visititems(visit)

print_tree(h5)

## 2. Root identity and storage metadata

The root attributes identify the sample, schema, reference, coordinate system, and P-site policy. The coverage datasets are compressed and chunked, so transcript slices can be read without loading the complete coordinate into memory.

In [ ]:
def display_value(value):
    if isinstance(value, bytes):
        return value.decode()
    if isinstance(value, np.ndarray):
        return [display_value(item) for item in value.tolist()]
    if isinstance(value, np.generic):
        return value.item()
    return value

root_attributes = pd.DataFrame(
    [(key, display_value(value)) for key, value in h5.attrs.items()],
    columns=["attribute", "value"],
).set_index("attribute")
display(root_attributes)

coverage_storage = []
for name, dataset in h5["coverage"].items():
    coverage_storage.append({
        "signal": name,
        "shape": dataset.shape,
        "dtype": str(dataset.dtype),
        "chunks": dataset.chunks,
        "compression": dataset.compression,
        "route": display_value(dataset.attrs.get("route", "")),
        "measure": display_value(dataset.attrs.get("measure", "")),
    })
display(pd.DataFrame(coverage_storage))

## 3. Find a transcript using a gene ID

The four coverage arrays concatenate every transcript in sorted transcript-ID order. `/transcripts/coverage_offset` and `/transcripts/transcript_len` locate one transcript inside those arrays. The lookup below accepts a versioned or unversioned gene ID and refuses to guess if multiple transcripts match.

In [ ]:
GENE_ID = "ENSG00000111640"  # GAPDH; replace with another Ensembl gene ID
TRANSCRIPT_ID = None            # set this if the gene lookup is ambiguous

def text_array(dataset):
    return np.asarray([display_value(value) for value in dataset[:]], dtype=object)

transcripts = h5["transcripts"]
gene_ids = text_array(transcripts["gene_id"])
transcript_ids = text_array(transcripts["transcript_id"])
gene_base = GENE_ID.split(".", 1)[0]
matches = np.flatnonzero([gene.split(".", 1)[0] == gene_base for gene in gene_ids])

if TRANSCRIPT_ID is not None:
    tx_base = TRANSCRIPT_ID.split(".", 1)[0]
    matches = np.asarray([
        index for index in matches
        if transcript_ids[index] == TRANSCRIPT_ID
        or transcript_ids[index].split(".", 1)[0] == tx_base
    ])

if len(matches) == 0:
    raise KeyError(f"No transcript found for gene {GENE_ID!r}")
if len(matches) > 1:
    candidates = transcript_ids[matches].tolist()
    raise ValueError(f"Gene {GENE_ID!r} has multiple transcripts: {candidates}. Set TRANSCRIPT_ID.")

transcript_index = int(matches[0])

def transcript_row(index):
    row = {}
    for name, dataset in transcripts.items():
        row[name] = display_value(dataset[index])
    return row

transcript = transcript_row(transcript_index)
display(pd.Series(transcript, name="value").to_frame())

## 4. Read the four tracks in the shared coordinate

Only the selected transcript slice is loaded. Every returned array has `transcript_len` positions, and position *i* denotes the same mature-transcript nucleotide for both alignment routes.

In [ ]:
signal_names = (
    "genome_psite",
    "txome_psite",
    "genome_footprint",
    "txome_footprint",
)
start = int(transcript["coverage_offset"])
length = int(transcript["transcript_len"])
stop = start + length
tracks = {name: h5["coverage"][name][start:stop] for name in signal_names}

track_summary = pd.DataFrame([
    {"signal": name, "positions": len(values), "sum": int(values.sum()), "max": int(values.max(initial=0))}
    for name, values in tracks.items()
])
display(track_summary)

## 5. Add the canonical UTR5/CDS/UTR3 overlay

In [ ]:
def rows_for_transcript(group_name, index):
    group = h5[group_name]
    indices = group["transcript_index"][:]
    rows = np.flatnonzero(indices == index)
    records = []
    for row_index in rows:
        records.append({name: display_value(dataset[row_index]) for name, dataset in group.items()})
    return pd.DataFrame(records)

regions = rows_for_transcript("regions", transcript_index)
display(regions[["label", "start", "end", "source"]])

In [ ]:
region_colors = {"UTR5": "#d9d9d9", "CDS": "#fee08b", "UTR3": "#d9d9d9"}
x = np.arange(length)
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True, constrained_layout=True)

panels = [
    ("P-site coverage", "genome_psite", "txome_psite"),
    ("Footprint coverage", "genome_footprint", "txome_footprint"),
]
for axis, (title, genome_signal, txome_signal) in zip(axes, panels):
    for region in regions.itertuples():
        axis.axvspan(region.start, region.end, color=region_colors.get(region.label, "#eeeeee"), alpha=0.35)
    axis.plot(x, tracks[genome_signal], color="#2166ac", linewidth=1, label="Genome alignment")
    axis.plot(x, tracks[txome_signal], color="#b2182b", linewidth=1, alpha=0.8, label="Transcriptome alignment")
    axis.set_ylabel("Coverage")
    axis.set_title(title)
    axis.legend(frameon=False)

axes[-1].set_xlabel("Transcript position (0-based, 5′→3′)")
fig.suptitle(f"{transcript['gene_name']} | {transcript['gene_id']} | {transcript['transcript_id']}")
plt.show()

## 6. Inspect the genomic-to-transcript exon map

`g_start:g_end` is a genomic, zero-based half-open interval. `tx_start:tx_end` is its corresponding interval in the mature transcript. Exons are ordered in biological 5′→3′ direction, including for negative-strand transcripts.

In [ ]:
exons = rows_for_transcript("exons", transcript_index)
display(exons[["exon_index", "chrom", "g_start", "g_end", "tx_start", "tx_end"]])

assert int((exons["g_end"] - exons["g_start"]).sum()) == length
assert int((exons["tx_end"] - exons["tx_start"]).sum()) == length
print(f"The exons tile all {length:,} transcript positions.")

## 7. Inspect the derived five-way ribosome-profile bins

These bins are a derived overlay and are intentionally separate from the canonical regions. In particular, the bin whose alias is `CDS` is not necessarily identical to the canonical annotated CDS.

In [ ]:
if "ribo_region_bins" in h5:
    ribo_bins = rows_for_transcript("ribo_region_bins", transcript_index)
    display(ribo_bins[["ribopy_alias", "label", "start", "end"]])
else:
    print("This file has no /ribo_region_bins group.")

## 8. Read the exact P-site offsets and provenance

The file records the read-length offsets used for each alignment route and a JSON provenance object describing its inputs, parameters, code identity, and generation command.

In [ ]:
for route in ("genome", "transcriptome"):
    group = h5[f"offsets/{route}"]
    table = pd.DataFrame({
        "read_length": group["read_length"][:],
        "psite_offset": group["psite_offset"][:],
    })
    print(route)
    display(table)

provenance = json.loads(display_value(h5["provenance"].attrs["json"]))
print(json.dumps(provenance, indent=2, sort_keys=True)[:8000])
if len(json.dumps(provenance)) > 8000:
    print("\n… output truncated for display")

## 9. Close the file

Downstream analysis needs only this HDF5. The BAMs and annotation inputs are needed to regenerate it, but not to inspect transcripts, calculate coverage concordance, or produce transcript-level coverage plots.

In [ ]:
h5.close()
print("Closed", COVERAGE_H5)